# AmbiFix lineage augmentation: Variant A first — RTX 5060 Ti 16 GB

**Prepared notebook, not completed BERT training.** Run the immutable conservative Variant A first. Variant B is optional and disabled by default. Each variant initializes a fresh six-class head and encoder from the same original pretrained `bert-base-uncased` commit; no historical fine-tuned AmbiFix model is an input.

The package keeps all 4,731 original training rows and the same ordered 1,015-row validation file. Variant A retains only accepted synthetic rows with no recorded review flag. Variant B retains the full accepted provisional set. **All synthetic labels remain UNVERIFIED; no flag does not guarantee correct meaning or ambiguity labels.** Original punctuation-key conflicts/overlaps are retained and documented, not silently corrected.

The historical validation macro F1 **0.5811396777932097** was verified from saved predictions. Its actual executed source, exact checkpoint/configuration and environment remain missing. This recipe follows the documented unbound reference as closely as possible, with explicit fixes/differences. Any difference from that historical score is descriptive exploratory validation evidence, not a controlled reproduction or final-test result.

Extract the complete ZIP, open this notebook from the package root or `notebooks/`, and run cells in order on the GPU collaborator's machine. Preserve the original input manifest. Return the executed notebook and complete output directories, including `best_model/`. Never add, inspect or evaluate the protected held-out file. Do not change data, flags, labels, split, seed or recipe after seeing validation scores.


## Public source release: local preparation is required

No research records, generated quality reports, model weights, or ready-made package manifest are distributed here. Run All deliberately fails closed until authorized local development inputs are restored and verified. Build variants using `scripts/build_variants.py` only after restoring the authorized lineage-generation outputs and original development partitions. Its actual passing dataset/lineage report must exist. Synthetic labels remain provisional.

From this experiment directory, run the source-only metadata helper after preparing those inputs:

```text
python scripts/prepare_local_manifest.py --root . --profile lineage --authorized-local-development-data --execute
```

This helper records input hashes only; it never copies records, packages a dataset, trains, downloads a model, or authorizes Test. It does not replace the notebook's unchanged data/label/weight/lineage checks. See `LOCAL_REPRODUCTION.md`.


## Optional GPU environment setup

Suggested pinned stack; its actual CUDA runtime has not been tested during local preparation. Commands are instructions, not automatically executed cells. Install only if needed, then restart the kernel. Start Python with `PYTHONHASHSEED=42` if feasible; setting it inside a running interpreter cannot retroactively ensure hash randomization.

```text
python -m pip install torch==2.9.0 --index-url https://download.pytorch.org/whl/cu128
python -m pip install transformers==4.51.3 numpy==1.26.4 pandas==2.2.3 scikit-learn==1.6.1 safetensors==0.5.3 huggingface_hub==0.36.2
```

The GPU preflight below requires approximately 16 GB, BF16 support, and a compatible compiled CUDA architecture. A failure must be returned with its logs rather than silently changing the experiment. Training uses BF16 autocast; **every validation forward pass runs in FP32 with autocast disabled**. Probability exports use FP64 softmax and lossless `.npy` files, plus 17-digit CSV values.


In [ ]:
# Small tabular/stdlib stack only. No Torch, CUDA or Hub imports before input checks.
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from collections import Counter
import hashlib
import json
import os
import platform
import random
import re
import subprocess
import sys
import traceback
import unicodedata
import warnings
import numpy as np
import pandas as pd

DATASET_CLASS_IDS = [1, 2, 3, 4, 5, 6]
EXPECTED_TRAIN_SHA256 = "c5d623c1738400170fe7b5feb23a82d4a3c15f071a77fe5c4bfba8c6c5cc70f8"
EXPECTED_VALIDATION_SHA256 = "9b5c7c76dad306bff4b306753d04a0169f84dd6f27eb8312bc63f5f3322bef42"
HISTORICAL_VALIDATION_MACRO_F1 = 0.5811396777932097
TRAIN_FILES = {"A": "data/train_A_Conservative_v1.csv", "B": "data/train_B_Full_Provisional_v1.csv"}
WEIGHT_FILES = {"A": "config/variant_A_class_weights.json", "B": "config/variant_B_class_weights.json"}
REQUIRED_HASHED_INPUTS = ["data/train_original.csv", "data/validation.csv", *TRAIN_FILES.values(),
    "config/training_config.json", *WEIGHT_FILES.values(), "reports/DATASET_VERIFICATION.json"]
PERMITTED_INPUTS = set(REQUIRED_HASHED_INPUTS + ["package_manifest.json"])
TRAIN_COLUMNS = ["source_id", "Text", "Class", "example_id", "parent_text", "pivot_language",
                 "is_synthetic", "label_review_status", "quality_review_flags"]
SYNTHETIC_METADATA = ["forward_model", "reverse_model", "forward_revision", "reverse_revision",
    "pivot_text", "seed", "max_input_tokens", "max_output_tokens", "num_beams", "generation_device", "generated_utc"]


def package_file(root, relative):
    if relative not in PERMITTED_INPUTS:
        raise ValueError("Notebook input is not allowlisted: " + str(relative))
    parts = PurePosixPath(relative).parts
    if PurePosixPath(relative).is_absolute() or ".." in parts or "\\" in relative or any(x.casefold()=="test.csv" for x in parts):
        raise ValueError("Unsafe or protected input path")
    path = (root / relative).resolve()
    if not path.is_relative_to(root.resolve()) or any(x.casefold()=="test.csv" for x in path.parts):
        raise ValueError("Input escapes the approved package or resolves to a protected path")
    return path


def manifest_hash_map(manifest):
    records = manifest.get("files")
    if not isinstance(records, list):
        raise ValueError("Manifest requires a files list of path/sha256 records")
    result = {}
    for item in records:
        relative, digest = item.get("path"), item.get("sha256")
        if not isinstance(relative,str) or not isinstance(digest,str):
            raise ValueError("Invalid manifest entry")
        parts=PurePosixPath(relative).parts
        if PurePosixPath(relative).is_absolute() or ".." in parts or "\\" in relative or any(x.casefold()=="test.csv" for x in parts):
            raise ValueError("Unsafe/protected manifest member")
        if relative in result or not re.fullmatch(r"[0-9a-fA-F]{64}",digest):
            raise ValueError("Duplicate path or invalid manifest digest")
        result[relative]=digest.lower()
    if not set(REQUIRED_HASHED_INPUTS).issubset(result):
        raise ValueError("Manifest is missing required training/validation/configuration/gate hashes")
    return result


def normalized(text):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC",str(text))).strip().casefold()


def punctuation_key(text):
    return re.sub(r"[\W_]+", "", normalized(text), flags=re.UNICODE)


def check_integer_classes(frame, name):
    if not {"source_id","Text","Class"}.issubset(frame.columns) or frame.empty:
        raise ValueError(name + ": missing columns or no rows")
    values=pd.to_numeric(frame.Class,errors="raise").to_numpy()
    integers=values.astype(int)
    if not np.array_equal(values,integers) or set(integers)!=set(DATASET_CLASS_IDS):
        raise ValueError(name + ": expected all six integer class IDs 1..6")
    if frame.source_id.astype(str).str.strip().eq("").any() or frame.Text.map(normalized).eq("").any():
        raise ValueError(name + ": blank source ID or Text")
    frame["Class"]=integers


def check_dataset_gate(gate, config, verified_hashes):
    if not str(gate.get("status","")).startswith("PASS") or gate.get("approved_for_gpu_package") is not True:
        raise ValueError("Dataset/lineage gate does not approve GPU packaging")
    if not isinstance(gate.get("checks"),dict) or not gate["checks"] or not all(x is True for x in gate["checks"].values()):
        raise ValueError("Dataset/lineage gate has a failed or missing check")
    if gate.get("protected_test_accessed") is not False:
        raise ValueError("Gate must explicitly record no protected-test access")
    if gate.get("benchmark_id") != config.get("benchmark_id") or not str(config.get("benchmark_id","")).strip():
        raise ValueError("Dataset gate belongs to a different experiment")
    if gate.get("validation_sha256")!=EXPECTED_VALIDATION_SHA256 or verified_hashes.get("data/validation.csv")!=EXPECTED_VALIDATION_SHA256:
        raise ValueError("Frozen validation bytes differ")
    if gate.get("train_original_sha256")!=EXPECTED_TRAIN_SHA256 or verified_hashes.get("data/train_original.csv")!=EXPECTED_TRAIN_SHA256:
        raise ValueError("Original training bytes differ")
    if "config_sha256" in gate and gate["config_sha256"]!=verified_hashes.get("config/training_config.json"):
        raise ValueError("Dataset gate is bound to a different training configuration")
    return True


def checked_config(config):
    fixed={"dataset_class_ids":DATASET_CLASS_IDS,"seed":42,"max_length":192,"train_batch_size":8,
        "eval_batch_size":16,"gradient_accumulation":2,"max_epochs":50,"min_epochs":8,
        "early_stop_patience":8,"min_delta":0.001,"learning_rate":2e-5,"weight_decay":0.01,
        "scheduler_patience":3,"scheduler_factor":0.5,"scheduler_min_lr":2e-7,
        "scheduler_threshold":0.001,"scheduler_threshold_mode":"rel","max_grad_norm":1.0,
        "class_weight_power":0.5,"gradient_checkpointing":False,"use_safetensors":True,
        "optimizer_name":"AdamW","optimizer_betas":[0.9,0.999],"optimizer_eps":1e-8,
        "tokenizer_use_fast":True,"dynamic_padding":True,"model_input_column":"Text",
        "attention_implementation":"eager","no_final_test_access":True,
        "train_precision":"bf16","validation_precision":"float32"}
    for key,value in fixed.items():
        if config.get(key)!=value:
            raise ValueError("Fixed recipe mismatch: " + key)
    if config.get("model_id") not in {"bert-base-uncased","google-bert/bert-base-uncased"}:
        raise ValueError("Only the original official BERT repository is allowed")
    if not re.fullmatch(r"[0-9a-f]{40}",str(config.get("pretrained_revision",""))):
        raise ValueError("Original pretrained revision must be an immutable commit")
    if config.get("dataset_to_model_index")!={str(i):i-1 for i in DATASET_CLASS_IDS}:
        raise ValueError("Dataset/model numeric label mapping changed")
    names=config.get("dataset_class_names",{})
    if set(names)!=set(map(str,DATASET_CLASS_IDS)) or len(set(names.values()))!=6:
        raise ValueError("Six distinct display names are required; numeric classes 2 and 4 must not merge")
    if config.get("historical_baseline_macro_f1")!=HISTORICAL_VALIDATION_MACRO_F1:
        raise ValueError("Historical saved-prediction reference changed")
    return config


def check_variants(original, validation, variants, original_count=4731, validation_count=1015, expected_full_synthetic=2443):
    original=original.copy();validation=validation.copy()
    check_integer_classes(original,"original train");check_integer_classes(validation,"validation")
    if len(original)!=original_count or len(validation)!=validation_count or not original.source_id.is_unique or not validation.source_id.is_unique:
        raise ValueError("Original/validation counts or IDs changed")
    parents=original.set_index("source_id").to_dict("index")
    if set(parents)&set(validation.source_id):
        raise ValueError("Original training and validation source IDs overlap")
    original_keys=set(original.Text.map(normalized));validation_keys=set(validation.Text.map(normalized))
    if len(original_keys)!=len(original) or len(validation_keys)!=len(validation) or original_keys&validation_keys:
        raise ValueError("Original/validation normalized duplication or overlap")
    original_punct=set(original.Text.map(punctuation_key));validation_punct=set(validation.Text.map(punctuation_key))
    result={};synthetic_ids={};original_triples=list(original[["source_id","Text","Class"]].itertuples(index=False,name=None))
    for variant,frame in variants.items():
        if not set(TRAIN_COLUMNS+SYNTHETIC_METADATA).issubset(frame.columns):
            raise ValueError(variant+": missing lineage/quality columns")
        check_integer_classes(frame,"variant "+variant)
        if not frame.example_id.is_unique or frame.example_id.astype(str).str.strip().eq("").any():
            raise ValueError(variant+": duplicate/blank example IDs")
        synth=pd.to_numeric(frame.is_synthetic,errors="raise").to_numpy()
        if not set(synth).issubset({0,1}):raise ValueError(variant+": invalid is_synthetic flag")
        old=frame.loc[synth==0];new=frame.loc[synth==1]
        triples=list(old[["source_id","Text","Class"]].itertuples(index=False,name=None))
        if triples!=original_triples or not np.all(synth[:original_count]==0) or np.any(synth[original_count:]==0):
            raise ValueError(variant+": original examples were removed, reordered or changed")
        if not old.example_id.eq(old.source_id).all() or not old.parent_text.eq(old.Text).all() or not old.label_review_status.eq("SOURCE_LABEL").all():
            raise ValueError(variant+": original lineage/status changed")
        if not new.label_review_status.eq("UNVERIFIED").all() or new.Class.eq(6).any():
            raise ValueError(variant+": synthetic labels must stay UNVERIFIED and Clean must not be oversampled")
        for row in frame.itertuples(index=False):
            parent=parents.get(row.source_id)
            if parent is None or row.parent_text!=parent["Text"] or row.Class!=parent["Class"]:
                raise ValueError(variant+": child parent/source label alignment changed")
        for field in SYNTHETIC_METADATA:
            if new[field].astype(str).str.strip().eq("").any():
                raise ValueError(variant+": missing synthetic metadata "+field)
        if not new.pivot_language.isin(["fr","de","es"]).all():
            raise ValueError(variant+": unknown synthetic pivot")
        if any(not re.fullmatch(r"[0-9a-f]{40}",str(r)) for field in ["forward_revision","reverse_revision"] for r in new[field]):
            raise ValueError(variant+": invalid translation revision")
        if frame.groupby("source_id").Class.nunique().gt(1).any():
            raise ValueError(variant+": parent group has conflicting labels")
        keys=frame.Text.map(normalized);new_keys=new.Text.map(normalized);new_punct=new.Text.map(punctuation_key)
        if keys.duplicated().any() or set(keys)&validation_keys or set(frame.source_id)&set(validation.source_id):
            raise ValueError(variant+": normalized/ID training-validation duplicate or overlap")
        if new_punct.duplicated().any() or set(new_punct)&(original_punct|validation_punct):
            raise ValueError(variant+": new synthetic punctuation duplication or overlap")
        flags=new.quality_review_flags.astype(str).str.strip()
        if variant=="A" and flags.ne("").any():raise ValueError("Variant A contains a flagged synthetic row")
        synthetic_ids[variant]=set(new.example_id)
        result[variant]={"training_rows":len(frame),"original_rows":len(old),"synthetic_rows":len(new),
            "flagged_synthetic_rows":int(flags.ne("").sum()),
            "training_source_groups":int(frame.source_id.nunique()),
            "class_counts":{str(k):int(v) for k,v in frame.Class.value_counts().sort_index().items()},
            "normalized_duplicate_rows":0,"normalized_validation_overlap":0,
            "new_synthetic_punctuation_overlap":0,
            "inherited_original_punctuation_duplicate_extra_rows":len(original)-len(original_punct),
            "inherited_original_validation_punctuation_shared_keys":len(original_punct&validation_punct)}
    if len(synthetic_ids["B"])!=expected_full_synthetic:raise ValueError("Variant B is not the complete accepted synthetic set")
    full_new=variants["B"].loc[pd.to_numeric(variants["B"].is_synthetic)==1]
    unflagged=set(full_new.loc[full_new.quality_review_flags.astype(str).str.strip().eq(""),"example_id"])
    if synthetic_ids["A"]!=unflagged:raise ValueError("Variant A differs from the exact unflagged subset of B")
    return result


def verified_weights(frame, record, power=0.5):
    counts=frame.Class.value_counts().reindex(DATASET_CLASS_IDS).to_numpy(dtype=np.int64)
    weights=(counts.sum()/(len(DATASET_CLASS_IDS)*counts.astype(np.float64)))**power
    weights/=weights.mean()
    if record.get("dataset_class_order")!=DATASET_CLASS_IDS or record.get("training_counts")!=counts.tolist():
        raise ValueError("Saved class-weight count/order differs from this training variant")
    supplied=np.asarray(record.get("values",[]),dtype=np.float64)
    if supplied.shape!=(6,) or not np.allclose(supplied,weights,rtol=0,atol=1e-12):
        raise ValueError("Saved class weights differ from training-only recomputation")
    return counts,weights


def check_cuda_build(capability,cuda_version,architectures):
    capability=tuple(int(x) for x in capability)
    if len(capability)!=2:raise ValueError("Invalid GPU capability")
    if capability[0]>=12:
        try:version=tuple(int(x) for x in str(cuda_version).split(".")[:2])
        except (ValueError,TypeError):raise ValueError("Cannot identify Blackwell CUDA build")
        if len(version)!=2 or version<(12,8):raise ValueError("This RTX recipe requires a CUDA 12.8+ build")
        required=f"sm_{capability[0]}{capability[1]}"
        if required not in architectures:raise ValueError("Installed wheel lacks compiled "+required)
    return {"compute_capability":list(capability),"cuda_build":cuda_version,"compiled_architectures":list(architectures)}


## Verify the immutable package before GPU use

This cell reads only explicit package-relative training, validation, configuration and verification inputs. It checks both variants so accidental changes fail before model loading. The inherited punctuation-key relationships are reported; the original examples are preserved. Unflagged synthetic rows remain unverified labels.


In [ ]:
_cwd=Path.cwd().resolve()
_candidates=[_cwd]+([_cwd.parent] if _cwd.name=="notebooks" else [])
_candidates=[p for p in _candidates if (p/"package_manifest.json").is_file() and (p/"config/training_config.json").is_file()]
if len(_candidates)!=1:raise FileNotFoundError("Open within the extracted package root or notebooks/ directory")
PACKAGE_ROOT=_candidates[0]
manifest=json.loads(package_file(PACKAGE_ROOT,"package_manifest.json").read_text(encoding="utf-8-sig"))
manifest_hashes=manifest_hash_map(manifest)
verified_input_hashes={}
for relative in REQUIRED_HASHED_INPUTS:
    actual=hashlib.sha256(package_file(PACKAGE_ROOT,relative).read_bytes()).hexdigest()
    if actual!=manifest_hashes[relative]:raise ValueError("Package hash mismatch: "+relative)
    verified_input_hashes[relative]=actual
cfg=checked_config(json.loads(package_file(PACKAGE_ROOT,"config/training_config.json").read_text(encoding="utf-8-sig")))
dataset_gate=json.loads(package_file(PACKAGE_ROOT,"reports/DATASET_VERIFICATION.json").read_text(encoding="utf-8-sig"))
check_dataset_gate(dataset_gate,cfg,verified_input_hashes)
BENCHMARK_ID=cfg["benchmark_id"]
original_df=pd.read_csv(package_file(PACKAGE_ROOT,"data/train_original.csv"),keep_default_na=False,dtype={"source_id":str})
val_df=pd.read_csv(package_file(PACKAGE_ROOT,"data/validation.csv"),keep_default_na=False,dtype={"source_id":str})
train_frames={v:pd.read_csv(package_file(PACKAGE_ROOT,path),keep_default_na=False,dtype={"source_id":str,"example_id":str}) for v,path in TRAIN_FILES.items()}
weight_records={v:json.loads(package_file(PACKAGE_ROOT,path).read_text(encoding="utf-8-sig")) for v,path in WEIGHT_FILES.items()}
integrity=check_variants(original_df,val_df,train_frames)
training_counts={};training_weights={}
for v in ["A","B"]:training_counts[v],training_weights[v]=verified_weights(train_frames[v],weight_records[v],cfg["class_weight_power"])
print("Experiment:",BENCHMARK_ID)
print(json.dumps(integrity,indent=2))
print("Validation: same ordered 1,015 source IDs and immutable byte hash",EXPECTED_VALIDATION_SHA256)
print("All synthetic labels remain UNVERIFIED. No quality flag is not semantic proof.")


## GPU preflight and common training functions

The configured original BERT commit is fixed before distribution. Each enabled variant receives fresh original pretrained weights and a freshly seeded classification head. No locally fine-tuned checkpoint is an initialization candidate. Training-only square-root weights are recomputed and checked separately for A and B.

Corrected weighted accumulation uses summed weighted cross entropy divided by the total target-class weight across the entire accumulation window, including its last partial window. This differs from the unbound reference's independently normalized microbatch averaging. Highest literal validation macro F1 selects `best_model/`; the 0.001 meaningful-improvement threshold controls the early-stop counter separately. The exact historical implementation is unavailable, so these corrections and FP32 validation remain comparison limitations.


In [ ]:
# Algorithm environment before CUDA initialization.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG",":4096:8")
import time
import torch
import transformers
import sklearn
from torch.utils.data import Dataset,DataLoader
from transformers import AutoTokenizer,AutoModelForSequenceClassification
from huggingface_hub import HfApi
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix

if not torch.cuda.is_available():raise RuntimeError("CUDA is required; this notebook must not train on the audit Mac")
device=torch.device("cuda")
gpu_name=torch.cuda.get_device_name(0)
gpu_gib=torch.cuda.get_device_properties(0).total_memory/1024**3
if gpu_gib<14:raise RuntimeError("This fixed recipe expects approximately 16 GB GPU memory")
if not torch.cuda.is_bf16_supported():raise RuntimeError("BF16 unsupported; return the failure rather than silently changing precision")
cuda_preflight=check_cuda_build(torch.cuda.get_device_capability(0),torch.version.cuda,torch.cuda.get_arch_list())
with torch.no_grad(),torch.autocast("cuda",dtype=torch.bfloat16):
    _probe=torch.randn(64,64,device=device)
    _product=_probe@_probe.T
    if not torch.isfinite(_product).all().item():raise FloatingPointError("CUDA smoke test produced nonfinite values")
    torch.cuda.synchronize()
    del _probe,_product
torch.cuda.empty_cache()
cuda_preflight["finite_bf16_matmul_passed"]=True
print("GPU:",gpu_name,"GiB:",round(gpu_gib,2),json.dumps(cuda_preflight))


def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
    torch.use_deterministic_algorithms(True,warn_only=True)


seed_all(cfg["seed"])
MODEL_ID=cfg["model_id"];PRETRAINED_REVISION=cfg["pretrained_revision"]
if Path(MODEL_ID).exists() or (PACKAGE_ROOT/MODEL_ID).exists():
    raise ValueError("A local initialization path would shadow the official pretrained repository")
pretrained_info=HfApi().model_info(MODEL_ID,revision=PRETRAINED_REVISION)
if pretrained_info.id not in {"bert-base-uncased","google-bert/bert-base-uncased"} or pretrained_info.sha!=PRETRAINED_REVISION:
    raise ValueError("Original pretrained repository/revision identity failed")
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=PRETRAINED_REVISION,use_fast=cfg["tokenizer_use_fast"])
ID2LABEL={i-1:cfg["dataset_class_names"][str(i)] for i in DATASET_CLASS_IDS}
LABEL2ID={value:key for key,value in ID2LABEL.items()}
OUTPUTS=PACKAGE_ROOT/"outputs"
OUTPUTS.mkdir(exist_ok=True)


class RequirementsDataset(Dataset):
    def __init__(self,frame):
        self.texts=frame.Text.astype(str).tolist();self.labels=(frame.Class.astype(int)-1).tolist()
    def __len__(self):return len(self.labels)
    def __getitem__(self,index):return self.texts[index],self.labels[index]


def collate(items):
    texts,labels=zip(*items)
    batch=tokenizer(list(texts),padding=True,truncation=True,max_length=cfg["max_length"],return_tensors="pt")
    batch["labels"]=torch.tensor(labels,dtype=torch.long)
    return batch


@torch.inference_mode()
def evaluate_fp32(current_model,loader,export_arrays=False):
    current_model.eval()
    if any(p.dtype!=torch.float32 for p in current_model.parameters()):
        raise TypeError("Validation requires FP32 model parameters")
    y_all=[];p_all=[];probability_parts=[];logit_parts=[];loss_sum=0.0;count=0
    for batch in loader:
        target=batch["labels"].to(device)
        features={key:value.to(device,non_blocking=True) for key,value in batch.items() if key!="labels"}
        # Do not convert BF16 forward logits to FP32 and call that full-precision inference.
        with torch.autocast(device_type="cuda",enabled=False):
            logits=current_model(**features).logits
        if logits.dtype!=torch.float32 or not torch.isfinite(logits).all().item():
            raise FloatingPointError("Validation logits must be finite FP32")
        loss=torch.nn.functional.cross_entropy(logits,target,reduction="sum")
        if not torch.isfinite(loss).item():raise FloatingPointError("Nonfinite validation loss")
        probability=torch.softmax(logits.to(dtype=torch.float64),dim=-1)
        if not torch.isfinite(probability).all().item():raise FloatingPointError("Nonfinite validation probabilities")
        prediction=probability.argmax(dim=-1)
        loss_sum+=float(loss.item());count+=len(target)
        y_all.extend(target.cpu().tolist());p_all.extend(prediction.cpu().tolist())
        if export_arrays:
            probability_parts.append(probability.cpu().numpy());logit_parts.append(logits.cpu().numpy())
    scores={"unweighted_loss":loss_sum/count,"accuracy":float(accuracy_score(y_all,p_all)),
        "macro_f1":float(f1_score(y_all,p_all,labels=list(range(6)),average="macro",zero_division=0)),
        "weighted_f1":float(f1_score(y_all,p_all,labels=list(range(6)),average="weighted",zero_division=0))}
    return scores,np.asarray(y_all,dtype=int),np.asarray(p_all,dtype=int),(
        np.concatenate(probability_parts) if export_arrays else None),(
        np.concatenate(logit_parts) if export_arrays else None)


def sha256_file(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda:handle.read(8*1024*1024),b""):digest.update(block)
    return digest.hexdigest()


def json_output(path,record):
    path.write_text(json.dumps(record,indent=2,ensure_ascii=False)+"\n",encoding="utf-8")


def completion_marker(variant):return OUTPUTS/f"COMPLETED_VARIANT_{variant}.json"


def load_completed_variant(variant):
    marker=completion_marker(variant)
    if not marker.is_file():return None
    saved=json.loads(marker.read_text(encoding="utf-8"))
    if saved.get("status")!="completed_validation_only" or saved.get("input_sha256")!=verified_input_hashes:
        raise ValueError("Existing completed run belongs to different immutable inputs")
    if saved.get("variant")!=variant or saved.get("train_dataset_sha256")!=verified_input_hashes[TRAIN_FILES[variant]]:
        raise ValueError("Completed-run variant/training dataset identity mismatch")
    if saved.get("configuration")!=cfg or saved.get("initialization",{}).get("revision")!=cfg["pretrained_revision"]:
        raise ValueError("Completed-run recipe or original pretrained revision mismatch")
    if saved.get("initialization",{}).get("repository") not in {"bert-base-uncased","google-bert/bert-base-uncased"}:
        raise ValueError("Completed-run original pretrained repository mismatch")
    directory=saved.get("run_directory","")
    if not isinstance(directory,str) or not directory or PurePosixPath(directory).is_absolute() or ".." in PurePosixPath(directory).parts or any(x.casefold()=="test.csv" for x in PurePosixPath(directory).parts):
        raise ValueError("Unsafe/protected completed-run directory")
    run_dir=(PACKAGE_ROOT/directory).resolve()
    if not run_dir.is_relative_to(OUTPUTS.resolve()) or any(x.casefold()=="test.csv" for x in run_dir.parts):
        raise ValueError("Completed-run directory escapes outputs or resolves to a protected path")
    artifacts=saved.get("completion_artifact_sha256")
    if not isinstance(artifacts,dict) or not artifacts:raise ValueError("Completed-run artifact hashes missing")
    for relative,digest in artifacts.items():
        if not isinstance(relative,str) or PurePosixPath(relative).is_absolute() or ".." in PurePosixPath(relative).parts or "\\" in relative or any(x.casefold()=="test.csv" for x in PurePosixPath(relative).parts):
            raise ValueError("Unsafe/protected completed artifact path")
        if not isinstance(digest,str) or not re.fullmatch(r"[0-9a-f]{64}",digest):raise ValueError("Invalid completed-artifact digest")
        path=(run_dir/relative).resolve()
        if not path.is_relative_to(run_dir) or any(x.casefold()=="test.csv" for x in path.parts):
            raise ValueError("Completed artifact escapes run or resolves to protected path")
        if sha256_file(path)!=digest:raise ValueError("Completed run artifact changed: "+relative)
    return saved


def run_variant_once(variant):
    if variant not in {"A","B"}:raise ValueError("Unknown variant")
    preceding_A=load_completed_variant("A") if variant=="B" else None
    if variant=="B" and preceding_A is None:
        raise RuntimeError("Variant A must complete before optional Variant B")
    existing=load_completed_variant(variant)
    if existing is not None:
        print("Using the actual existing completed Variant",variant,"without retraining:",existing["run_directory"])
        return existing
    marker_started=OUTPUTS/f"STARTED_VARIANT_{variant}.json"
    if marker_started.exists():
        raise RuntimeError("This variant has an incomplete/failed started run. Return its real logs; do not automatically rerun after seeing validation results.")
    stamp=datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f_UTC")
    run_dir=OUTPUTS/f"variant_{variant}_sqrtweighted_{stamp}"
    run_dir.mkdir(exist_ok=False)
    with marker_started.open("x",encoding="utf-8") as handle:
        json.dump({"variant":variant,"run_directory":str(run_dir.relative_to(PACKAGE_ROOT)),"status":"started","input_sha256":verified_input_hashes},handle,indent=2)
    started=time.time();caught_warnings=[];history=[];model=None;best_model=None;completed_result=None
    best_macro=-float("inf");best_epoch=None;meaningful_macro=-float("inf");meaningful_epoch=None;stale=0
    frame=train_frames[variant]
    counts,weights64=verified_weights(frame,weight_records[variant],cfg["class_weight_power"])
    json_output(run_dir/"training_config.json",cfg)
    json_output(run_dir/"input_integrity.json",integrity)
    json_output(run_dir/"input_dataset_gate.json",dataset_gate)
    json_output(run_dir/"class_weights.json",{**weight_records[variant],"variant":variant,"source_partition":"this variant training rows only",
        "training_counts":counts.tolist(),"values":weights64.tolist(),"loss_weight_tensor_dtype":"float32",
        "loss_reduction":"summed weighted CE gradients normalized by target-class-weight sum across the whole accumulation window"})
    best_dir=run_dir/"best_model"
    try:
        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter("always")
            seed_all(cfg["seed"])
            generator=torch.Generator().manual_seed(cfg["seed"])
            train_loader=DataLoader(RequirementsDataset(frame),batch_size=cfg["train_batch_size"],shuffle=True,generator=generator,num_workers=0,pin_memory=True,collate_fn=collate)
            val_loader=DataLoader(RequirementsDataset(val_df),batch_size=cfg["eval_batch_size"],shuffle=False,num_workers=0,pin_memory=True,collate_fn=collate)
            # Reset immediately before every fresh original-pretrained initialization.
            seed_all(cfg["seed"])
            model,loading_info=AutoModelForSequenceClassification.from_pretrained(
                MODEL_ID,revision=PRETRAINED_REVISION,num_labels=6,id2label=ID2LABEL,label2id=LABEL2ID,
                use_safetensors=cfg["use_safetensors"],attn_implementation=cfg["attention_implementation"],output_loading_info=True)
            if model.config.model_type!="bert":raise ValueError("Unexpected architecture")
            if any(key.startswith("bert.embeddings.") or key.startswith("bert.encoder.") for key in loading_info.get("missing_keys",[])):
                raise ValueError("Original pretrained encoder weights are missing")
            json_output(run_dir/"pretrained_loading_info.json",loading_info)
            model=model.float().to(device)
            model.gradient_checkpointing_disable()
            if model.is_gradient_checkpointing:raise RuntimeError("Gradient checkpointing should remain disabled")
            model.config.to_json_file(run_dir/"initial_model_config.json")
            optimizer=torch.optim.AdamW(model.parameters(),lr=cfg["learning_rate"],weight_decay=cfg["weight_decay"],betas=tuple(cfg["optimizer_betas"]),eps=cfg["optimizer_eps"])
            scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode="max",factor=cfg["scheduler_factor"],
                patience=cfg["scheduler_patience"],threshold=cfg["scheduler_threshold"],threshold_mode=cfg["scheduler_threshold_mode"],min_lr=cfg["scheduler_min_lr"])
            weights=torch.tensor(weights64,dtype=torch.float32,device=device)
            for epoch in range(1,cfg["max_epochs"]+1):
                model.train();optimizer.zero_grad(set_to_none=True)
                numerator_total=0.0;denominator_total=0.0;window_denominator=0.0;optimizer_steps=0
                epoch_started=time.time()
                for batch_number,batch in enumerate(train_loader,start=1):
                    target=batch["labels"].to(device)
                    features={key:value.to(device,non_blocking=True) for key,value in batch.items() if key!="labels"}
                    with torch.autocast(device_type="cuda",dtype=torch.bfloat16):
                        logits=model(**features).logits
                    if not torch.isfinite(logits).all().item():raise FloatingPointError("Nonfinite training logits")
                    numerator=torch.nn.functional.cross_entropy(logits.float(),target,weight=weights,reduction="sum")
                    denominator=float(weights[target].sum().item())
                    if not torch.isfinite(numerator).item() or denominator<=0:raise FloatingPointError("Invalid weighted training loss")
                    numerator.backward()
                    numerator_total+=float(numerator.item());denominator_total+=denominator;window_denominator+=denominator
                    if batch_number%cfg["gradient_accumulation"]==0 or batch_number==len(train_loader):
                        for parameter in model.parameters():
                            if parameter.grad is not None:parameter.grad.div_(window_denominator)
                        torch.nn.utils.clip_grad_norm_(model.parameters(),cfg["max_grad_norm"],error_if_nonfinite=True)
                        optimizer.step();optimizer_steps+=1;optimizer.zero_grad(set_to_none=True);window_denominator=0.0
                scores,_,_,_,_=evaluate_fp32(model,val_loader)
                if not all(np.isfinite(v) for v in scores.values()):raise FloatingPointError("Nonfinite validation metric")
                lr_before=optimizer.param_groups[0]["lr"]
                scheduler.step(scores["macro_f1"])
                improved_literal=scores["macro_f1"]>best_macro
                improved_meaningful=scores["macro_f1"]>meaningful_macro+cfg["min_delta"]
                if improved_meaningful:meaningful_macro=scores["macro_f1"];meaningful_epoch=epoch;stale=0
                else:stale+=1
                row={"epoch":epoch,"train_weighted_loss":numerator_total/denominator_total,
                    **{"validation_"+key:value for key,value in scores.items()},"optimizer_steps":optimizer_steps,
                    "lr_before_scheduler":lr_before,"lr_after_scheduler":optimizer.param_groups[0]["lr"],
                    "literal_best_updated":improved_literal,"meaningful_improvement":improved_meaningful,
                    "early_stopping_stale_epochs":stale,"epoch_seconds":time.time()-epoch_started}
                history.append(row)
                pd.DataFrame(history).to_csv(run_dir/"epoch_metrics.csv",index=False,float_format="%.17g")
                with (run_dir/"training_log.jsonl").open("a",encoding="utf-8") as handle:handle.write(json.dumps(row)+"\n")
                if improved_literal:
                    best_macro=scores["macro_f1"];best_epoch=epoch
                    best_dir.mkdir(exist_ok=True);model.save_pretrained(best_dir,safe_serialization=True);tokenizer.save_pretrained(best_dir)
                    json_output(run_dir/"best_epoch.json",{"selected_epoch":best_epoch,"selected_macro_f1":best_macro,
                        "selection_rule":"strictly highest literal validation macro F1; earliest epoch wins a tie",
                        "meaningful_improvement_epoch":meaningful_epoch,"meaningful_improvement_macro_f1":meaningful_macro,
                        "minimum_improvement_for_early_stop_counter_only":cfg["min_delta"]})
                    torch.save({"epoch":epoch,"optimizer":optimizer.state_dict(),"scheduler":scheduler.state_dict(),
                        "python_rng_state":random.getstate(),"numpy_rng_state":np.random.get_state(),"torch_rng_state":torch.get_rng_state(),
                        "cuda_rng_states":torch.cuda.get_rng_state_all(),"dataloader_generator_state":generator.get_state(),
                        "input_sha256":verified_input_hashes,"configuration":cfg},run_dir/"best_training_state.pt")
                print(f"Variant {variant} epoch {epoch:02d}: macro={scores['macro_f1']:.6f} weighted={scores['weighted_f1']:.6f} accuracy={scores['accuracy']:.6f}; best epoch {best_epoch}",flush=True)
                if epoch>=cfg["min_epochs"] and stale>=cfg["early_stop_patience"]:
                    print("Meaningful-improvement early stopping; literal best checkpoint remains epoch",best_epoch);break
            if best_epoch is None:raise RuntimeError("No checkpoint selected")
            del model,optimizer,scheduler,weights
            model=None;torch.cuda.empty_cache()
            best_model=AutoModelForSequenceClassification.from_pretrained(best_dir,use_safetensors=True,attn_implementation=cfg["attention_implementation"]).float().to(device)
            final,y0,p0,probabilities,logits=evaluate_fp32(best_model,val_loader,export_arrays=True)
            if abs(final["macro_f1"]-best_macro)>1e-12:raise RuntimeError("Reloaded best checkpoint does not reproduce selected macro F1")
            true_labels=y0+1;predicted_labels=p0+1
            if not np.array_equal(true_labels,val_df.Class.to_numpy(dtype=int)):raise ValueError("Frozen validation order/labels changed")
            if probabilities.dtype!=np.float64 or logits.dtype!=np.float32 or not np.isfinite(probabilities).all() or not np.allclose(probabilities.sum(axis=1),1,rtol=0,atol=1e-12):
                raise FloatingPointError("Invalid full-precision probability/logit export")
            if not np.array_equal(probabilities.argmax(axis=1)+1,predicted_labels):raise ValueError("Exported class order mismatch")
            np.save(run_dir/"validation_probabilities_float64.npy",probabilities,allow_pickle=False)
            np.save(run_dir/"validation_logits_float32.npy",logits,allow_pickle=False)
            predictions=val_df[["source_id","Text","Class"]].copy();predictions["predicted_class"]=predicted_labels
            predictions["correct"]=true_labels==predicted_labels
            for j,label in enumerate(DATASET_CLASS_IDS):
                predictions[f"prob_class_{label}"]=probabilities[:,j];predictions[f"logit_class_{label}"]=logits[:,j]
            predictions.to_csv(run_dir/"validation_predictions.csv",index=False,encoding="utf-8-sig",float_format="%.17g")
            predictions.loc[~predictions.correct].to_csv(run_dir/"validation_errors.csv",index=False,encoding="utf-8-sig",float_format="%.17g")
            report=classification_report(true_labels,predicted_labels,labels=DATASET_CLASS_IDS,output_dict=True,zero_division=0)
            json_output(run_dir/"classification_report.json",report)
            (run_dir/"classification_report.txt").write_text(classification_report(true_labels,predicted_labels,labels=DATASET_CLASS_IDS,digits=10,zero_division=0),encoding="utf-8")
            cm=confusion_matrix(true_labels,predicted_labels,labels=DATASET_CLASS_IDS)
            pd.DataFrame(cm,index=[f"true_{i}" for i in DATASET_CLASS_IDS],columns=[f"pred_{i}" for i in DATASET_CLASS_IDS]).to_csv(run_dir/"confusion_matrix.csv")
            metrics={"benchmark_id":BENCHMARK_ID,"variant":variant,"scope":"same frozen validation; exploratory selection, no held-out evaluation",
                "validation_rows":len(val_df),"selected_epoch":best_epoch,"validation_precision":"FP32 forward, FP64 softmax probabilities",
                **{"validation_"+key:value for key,value in final.items()},"dataset_class_order":DATASET_CLASS_IDS,
                "historical_saved_prediction_macro_f1":HISTORICAL_VALIDATION_MACRO_F1,
                "descriptive_macro_f1_delta_vs_historical":final["macro_f1"]-HISTORICAL_VALIDATION_MACRO_F1,
                "comparison_limit":"Historical executed configuration/checkpoint/environment are unavailable; this is not a controlled reproduction or causal augmentation-effect estimate",
                "synthetic_labels_expert_verified":False,"final_test_evaluation_performed":False}
            json_output(run_dir/"validation_metrics.json",metrics)
            with (run_dir/"pip_freeze.txt").open("w",encoding="utf-8") as handle:subprocess.run([sys.executable,"-m","pip","freeze"],stdout=handle,check=True)
            checkpoint_hashes={str(p.relative_to(run_dir)):sha256_file(p) for p in sorted(best_dir.iterdir()) if p.is_file()}
            provenance={"status":"completed_validation_only","benchmark_id":BENCHMARK_ID,"variant":variant,
                "run_directory":str(run_dir.relative_to(PACKAGE_ROOT)),"configuration":cfg,"input_sha256":verified_input_hashes,
                "preceding_completed_A_run":None if preceding_A is None else preceding_A["run_directory"],
                "train_dataset_sha256":verified_input_hashes[TRAIN_FILES[variant]],"training_rows":len(frame),
                "training_counts":counts.tolist(),"class_weight_values":weights64.tolist(),"integrity":integrity[variant],
                "label_mapping_dataset_to_model":{str(i):i-1 for i in DATASET_CLASS_IDS},
                "initialization":{"repository":pretrained_info.id,"revision":PRETRAINED_REVISION,"historical_finetuned_initialization_used":False},
                "selected_epoch":best_epoch,"highest_logged_macro_f1":max(r["validation_macro_f1"] for r in history),
                "meaningful_improvement_epoch":meaningful_epoch,"epochs_completed":len(history),"selected_validation_metrics":final,
                "checkpoint_sha256":checkpoint_hashes,"prepared_notebook_sha256":manifest_hashes.get("notebooks/AmbiFix_Lineage_Augmentation_BERT_A_First_RTX5060Ti.ipynb"),"executed_notebook_must_be_returned":True,"class_weights_source":"variant training labels only",
                "gpu":gpu_name,"gpu_gib":gpu_gib,"cuda_preflight":cuda_preflight,"python":sys.version,"platform":platform.platform(),
                "torch":torch.__version__,"transformers":transformers.__version__,"sklearn":sklearn.__version__,"numpy":np.__version__,"pandas":pd.__version__,
                "cuda":torch.version.cuda,"pythonhashseed_environment":os.environ.get("PYTHONHASHSEED"),
                "precision":{"training":"BF16 autocast, FP32 parameters/loss","validation":"FP32 forward with autocast disabled","saved_probabilities":"FP64 softmax and lossless NPY plus17-digit CSV"},
                "determinism":"Fixed seeds and deterministic-algorithm warnings; cross-runtime/hardware bitwise identity not guaranteed",
                "synthetic_labels_expert_verified":False,"test_data_read":False,"held_out_evaluation_performed":False,
                "further_hyperparameter_search_performed":False,"wall_seconds":time.time()-started,
                "baseline_comparison_limit":metrics["comparison_limit"]}
            json_output(run_dir/"experiment_provenance.json",provenance)
            required_completion=["validation_metrics.json","validation_predictions.csv","validation_probabilities_float64.npy","validation_logits_float32.npy",
                "experiment_provenance.json","epoch_metrics.csv","class_weights.json","best_training_state.pt","training_config.json","input_integrity.json",
                "input_dataset_gate.json","best_epoch.json","classification_report.json","classification_report.txt","confusion_matrix.csv","training_log.jsonl",
                "pretrained_loading_info.json","initial_model_config.json","pip_freeze.txt",*checkpoint_hashes]
            completion={**provenance,"completion_artifact_sha256":{relative:sha256_file(run_dir/relative) for relative in required_completion}}
            json_output(run_dir/"run_status.json",{"status":"completed_validation_only","variant":variant,"selected_epoch":best_epoch})
            completed_result=completion
            print("Completed actual Variant",variant,"validation evidence:",run_dir)
            print(json.dumps(metrics,indent=2))
            return completion
    except BaseException as error:
        completed_result=None
        json_output(run_dir/"run_status.json",{"status":"failed_or_interrupted","exception_type":type(error).__name__,"message":str(error),"variant":variant})
        (run_dir/"failure_traceback.txt").write_text(traceback.format_exc(),encoding="utf-8")
        raise
    finally:
        json_output(run_dir/"python_warnings.json",[{"category":w.category.__name__,"message":str(w.message),"filename":w.filename,"lineno":w.lineno} for w in caught_warnings])
        if model is not None:del model
        if best_model is not None:del best_model
        torch.cuda.empty_cache()
        if completed_result is not None:
            completed_result["completion_artifact_sha256"]["python_warnings.json"]=sha256_file(run_dir/"python_warnings.json")
            completed_result["completion_artifact_sha256"]["run_status.json"]=sha256_file(run_dir/"run_status.json")
            json_output(completion_marker(variant),completed_result)


## Run Variant A once

This cell starts the conservative variant only when executed on the collaborator's GPU. It returns an already completed, hash-checked A result without retraining if the cell is accidentally rerun. An interrupted/failed started run is retained and fails closed; return its actual logs instead of silently deleting it and starting another validation-selected trial.


In [ ]:
A_RESULT = run_variant_once("A")


## Optional Variant B — disabled by default

Prioritize Variant A before the November 2 deadline. If time permits, set the single switch below to `True` deliberately. B requires a completed, hash-verified A run and starts independently from the same original pretrained commit and seed; it never warm-starts from A. It uses the full provisional training data and its separately computed weights. Do not change the frozen data or recipe to increase validation F1.


In [ ]:
RUN_VARIANT_B = False
if RUN_VARIANT_B:
    B_RESULT = run_variant_once("B")
else:
    print("Variant B remains optional and was not trained by this cell.")


## Return the actual evidence

Return this executed notebook, original package manifest, completion marker(s), and every enabled run's full directory including `best_model/`, `best_training_state.pt`, tokenizer/configuration, training-only weights, epoch/log files, environment, selected-epoch metadata, classification/confusion reports, row-aligned predictions, FP32 logits and FP64 probabilities in lossless NPY form. Retain any actual error/warning/partial outputs.

A completed notebook preparation or package is not a completed experiment. No model scores are supplied for A or B until the GPU run produces the actual artifacts. Synthetic labels remain UNVERIFIED. The prior 0.581140 score is historical saved-prediction evidence; differences are exploratory because the historical training configuration is not recovered. The protected held-out dataset must never be added or evaluated in this workflow.
